# 05 — Analysis and Publication-Ready Plots

Generates all plots and statistical analyses from benchmark results.

**Prerequisites:** Run the benchmark first:
```bash
python scripts/run_benchmark.py --dataset all --phase zero_shot --save-probas
python scripts/run_scaling_test.py --dataset give_me_credit
```

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.visualization.plots import (
    plot_leaderboard, plot_scaling_curves, plot_calibration_diagrams,
    plot_limitation_matrix, plot_time_vs_accuracy, set_style,
)
from src.evaluation.metrics import compute_ranks, friedman_test, wilcoxon_signed_rank

RESULTS_DIR = Path('../results')
FIGURES_DIR = RESULTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

set_style()
print('Setup complete.')
print(f'Results dir: {RESULTS_DIR.resolve()}')

## 1. Load All Benchmark Results

In [ ]:
# Load zero-shot results for all datasets
zero_shot_dfs = {}
for csv_path in sorted(RESULTS_DIR.glob('zero_shot_*.csv')):
    dataset_name = csv_path.stem.replace('zero_shot_', '')
    df = pd.read_csv(csv_path)
    zero_shot_dfs[dataset_name] = df
    n_success = df['success'].sum()
    print(f'{dataset_name}: {n_success}/{len(df)} models succeeded')

# Load scaling results
scaling_dfs = {}
for csv_path in sorted(RESULTS_DIR.glob('scaling_*.csv')):
    dataset_name = csv_path.stem.replace('scaling_', '')
    scaling_dfs[dataset_name] = pd.read_csv(csv_path)

print(f'\nLoaded: {len(zero_shot_dfs)} zero-shot datasets, {len(scaling_dfs)} scaling datasets')

## 2. Performance Leaderboards

In [ ]:
for dataset_name, df in zero_shot_dfs.items():
    if df['success'].sum() == 0:
        print(f'Skipping {dataset_name}: no successful results.')
        continue
    plot_leaderboard(
        df,
        metric='auc_roc',
        title=f'Zero-Shot Performance — {dataset_name.replace("_", " ").title()}',
        save_path=str(FIGURES_DIR / f'leaderboard_{dataset_name}.png'),
    )
    plt.show()

## 3. TFM vs GBDT Summary Table

In [ ]:
TFM_MODELS = [
    'TabPFN-v1', 'TabPFN-v2', 'TabPFN-v2.5', 'Real-TabPFN-2.5',
    'TabICL-v2', 'TabICL-v1.1', 'Mitra', 'TabDPT',
]
GBDT_MODELS = [
    'XGBoost-Default', 'XGBoost-Tuned',
    'CatBoost-Default', 'CatBoost-Tuned',
    'LightGBM-Default',
]

comparison_rows = []
for dataset_name, df in zero_shot_dfs.items():
    df_ok = df[df['success']]
    if df_ok.empty:
        continue
    
    tfm_df = df_ok[df_ok['model_name'].isin(TFM_MODELS)]
    gbdt_df = df_ok[df_ok['model_name'].isin(GBDT_MODELS)]
    
    if not tfm_df.empty and not gbdt_df.empty:
        best_tfm = tfm_df.loc[tfm_df['auc_roc'].idxmax()]
        best_gbdt = gbdt_df.loc[gbdt_df['auc_roc'].idxmax()]
        comparison_rows.append({
            'Dataset': dataset_name,
            'Best TFM': best_tfm['model_name'],
            'TFM AUC': f"{best_tfm['auc_roc']:.4f}",
            'Best GBDT': best_gbdt['model_name'],
            'GBDT AUC': f"{best_gbdt['auc_roc']:.4f}",
            'TFM wins?': 'YES' if best_tfm['auc_roc'] > best_gbdt['auc_roc'] else 'no',
        })

if comparison_rows:
    print('TFM vs Best GBDT Comparison:')
    pd.DataFrame(comparison_rows).set_index('Dataset')
else:
    print('Not enough results. Run: python scripts/run_benchmark.py --dataset all --phase zero_shot')

## 4. Scaling Curves

In [ ]:
for dataset_name, df in scaling_dfs.items():
    if df.empty:
        continue
    plot_scaling_curves(
        df,
        title=f'Scaling Experiment — {dataset_name.replace("_", " ").title()}',
        save_path=str(FIGURES_DIR / f'scaling_{dataset_name}.png'),
    )
    plt.show()

## 5. Time vs Accuracy Pareto Front

In [ ]:
for dataset_name, df in zero_shot_dfs.items():
    df_ok = df[df['success'] & df['total_time'].notna()]
    if df_ok.empty:
        continue
    plot_time_vs_accuracy(
        df_ok,
        title=f'Time vs Accuracy — {dataset_name.replace("_", " ").title()}',
        save_path=str(FIGURES_DIR / f'time_vs_acc_{dataset_name}.png'),
    )
    plt.show()

## 6. Model Limitation Heatmap

In [ ]:
# Collect all results across datasets
if zero_shot_dfs:
    all_results = pd.concat(list(zero_shot_dfs.values()), ignore_index=True)
    plot_limitation_matrix(
        all_results,
        save_path=str(FIGURES_DIR / 'limitation_matrix.png'),
    )
    plt.show()
else:
    print('No results loaded.')

## 7. Statistical Tests

In [ ]:
if len(zero_shot_dfs) >= 2:
    # Compute average ranks across datasets
    all_dfs = list(zero_shot_dfs.values())
    ranks = compute_ranks(all_dfs, metric='auc_roc')
    
    print('Average Model Rankings (lower = better):')
    print(pd.Series(ranks).sort_values().to_string())
    
    # Friedman test (is there a significant difference among models?)
    try:
        friedman_result = friedman_test(all_dfs, metric='auc_roc')
        print(f'\nFriedman test: statistic={friedman_result["statistic"]:.3f}, p={friedman_result["p_value"]:.4f}')
        if friedman_result['p_value'] < 0.05:
            print('  -> Significant differences exist among models (p < 0.05)')
        else:
            print('  -> No significant difference detected (p >= 0.05)')
    except Exception as e:
        print(f'Statistical test failed: {e}')
else:
    print('Need results for at least 2 datasets for statistical tests.')
    print('Run: python scripts/run_benchmark.py --dataset all')

## 8. Export LaTeX Table

In [ ]:
# Create a publication-ready comparison table
table_rows = []
for dataset_name, df in zero_shot_dfs.items():
    df_ok = df[df['success']].copy()
    if df_ok.empty:
        continue
    df_ok = df_ok.sort_values('auc_roc', ascending=False)
    for _, row in df_ok.iterrows():
        table_rows.append({
            'Model': row['model_name'],
            'Dataset': dataset_name,
            'AUC-ROC': f"{row['auc_roc']:.4f}",
            'LogLoss': f"{row['log_loss_val']:.4f}",
            'ECE': f"{row['ece']:.4f}",
            'Time(s)': f"{row['total_time']:.1f}",
        })

if table_rows:
    table_df = pd.DataFrame(table_rows)
    pivot = table_df.pivot_table(
        index='Model', columns='Dataset', values='AUC-ROC', aggfunc='first'
    )
    print('AUC-ROC across datasets:')
    print(pivot)
    
    # Export to LaTeX
    latex_path = RESULTS_DIR / 'benchmark_table.tex'
    pivot.to_latex(latex_path, caption='Zero-Shot AUC-ROC Comparison', label='tab:benchmark')
    print(f'\nLaTeX table saved: {latex_path}')
else:
    print('No results available to export.')